# 02 - K-Means on a Subsample  (Stage 2)

**Purpose.** Fit `MiniBatchKMeans` on a 200k subsample of `complete`
trajectories for several `k`, evaluate each `k` (inertia, silhouette, cluster
sizes, maps), and pickle every model so the best `k` can be chosen later.

**Input.** `data/features.parquet`  (Stage 1).
**Output.** `data/kmeans_models/kmeans_k*.pkl`, evaluation plots in `figures/`.

**Feature handling.** Positions are *not* standardised (absolute geography
matters). With `config.FEATURE_SPACE = "xyz"` (default) each day-50 / day-100
position is embedded on the unit sphere as Cartesian `(x, y, z)` inside
`pipeline.build_feature_matrix`, giving a 6-D vector. Euclidean distance there is
the chord -- a monotonic function of the great-circle (haversine) distance -- so
plain k-means clusters by true spherical distance and the centroids
(re-projected to lat/lon by `pipeline.centroids_to_degrees`) are proper spherical
centroids. The `z` here is the polar geometry axis, **not** ocean depth. Set
`FEATURE_SPACE = "cosine"` for the legacy near-equator approximation.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)

project root: /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmeans_50_100days


## 2.1  Load features and take a reproducible 200k subsample

In [2]:
features = pd.read_parquet(C.FEATURES_FILE)
complete = features[features.status == "complete"]
n_sub = min(C.N_SUB, len(complete))
sub = complete.sample(n=n_sub, random_state=C.RANDOM_STATE)
X_sub = P.build_feature_matrix(sub)
print(f"complete: {len(complete):,}  ->  subsample: {X_sub.shape}")

complete: 15,880,000  ->  subsample: (200000, 6)


## 2.2  Fit MiniBatchKMeans for each k and evaluate

In [3]:
import pickle, time
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

results = []
for k in C.K_LIST:
    t0 = time.time()
    km = MiniBatchKMeans(n_clusters=k, random_state=C.RANDOM_STATE,
                         batch_size=10000, n_init=10, max_iter=300)
    labels = km.fit_predict(X_sub)
    sil = silhouette_score(X_sub, labels, sample_size=C.SILHOUETTE_SAMPLE,
                           random_state=C.RANDOM_STATE)
    sizes = np.bincount(labels, minlength=k)
    with open(C.MODELS_DIR / f"kmeans_k{k}.pkl", "wb") as f:
        pickle.dump(km, f)
    results.append(dict(k=k, inertia=km.inertia_, silhouette=sil,
                        min_size=sizes.min(), max_size=sizes.max()))
    print(f"k={k:2d}  inertia={km.inertia_:.1f}  silhouette={sil:.3f}  "
          f"sizes[{sizes.min()}..{sizes.max()}]  ({time.time()-t0:.1f}s)")

res = pd.DataFrame(results)
res

k=15  inertia=300.1  silhouette=0.392  sizes[2254..59886]  (27.2s)


k=20  inertia=256.2  silhouette=0.358  sizes[964..61368]  (27.1s)


k=25  inertia=217.5  silhouette=0.400  sizes[1519..46002]  (26.3s)


k=30  inertia=193.8  silhouette=0.397  sizes[1230..46101]  (26.4s)


k=35  inertia=183.1  silhouette=0.355  sizes[1185..31346]  (25.9s)


,k,inertia,silhouette,min_size,max_size
0,15,300.116215,0.392295,2254,59886
1,20,256.164544,0.358026,964,61368
2,25,217.522165,0.400126,1519,46002
3,30,193.798016,0.396741,1230,46101
4,35,183.053062,0.355265,1185,31346


## 2.3  Elbow and silhouette plots

In [4]:
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(res.k, res.inertia, "o-"); a1.set_xlabel("k"); a1.set_ylabel("inertia")
a1.set_title("Elbow plot")
a2.plot(res.k, res.silhouette, "o-"); a2.set_xlabel("k")
a2.set_ylabel("silhouette"); a2.set_title("Silhouette vs k")
fig.tight_layout()
fig.savefig(C.FIG_DIR / "elbow_plot.png", dpi=130)
fig.savefig(C.FIG_DIR / "silhouette_plot.png", dpi=130)
plt.show()

## 2.4  Cluster maps for each k

For every `k`, colour the subsample's day-50 (dots) and day-100 (x) positions by
cluster label so the spatial structure can be judged by eye.

In [5]:
import pickle
fig, axes = plt.subplots(1, len(C.K_LIST), figsize=(4*len(C.K_LIST), 4), squeeze=False)
for ax, k in zip(axes[0], C.K_LIST):
    with open(C.MODELS_DIR / f"kmeans_k{k}.pkl", "rb") as f:
        km = pickle.load(f)
    lab = km.predict(X_sub)
    ax.scatter(sub.lon_50, sub.lat_50, c=lab, s=2, cmap="tab20", alpha=.4)
    ax.scatter(sub.lon_100, sub.lat_100, c=lab, s=2, cmap="tab20", marker="x", alpha=.4)
    ax.set_title(f"k={k}"); ax.set_xlabel("lon"); ax.set_ylabel("lat")
fig.tight_layout()
fig.savefig(C.FIG_DIR / "cluster_maps_by_k.png", dpi=120)
plt.show()

## 2.5  Suggested cluster groupings (over-resolve, then merge)

We are **not** looking for a single "optimal" k. Instead pick a fairly high k
(e.g. 20-25) so the structure is over-resolved into many fine clusters, then
merge the fine clusters that represent the *same* physical pathway into a few
named groups.

The dendrogram below performs a Ward hierarchical clustering on the chosen
model's centroids: clusters that join low down are spatially similar and are
good candidates to merge. `pipeline.suggest_groups` cuts that tree into
`n_groups` and returns a `{raw_cluster: group}` map you can paste into
`config.GROUP_MAP` (and then refine by eye).

In [6]:
import pickle
from scipy.cluster.hierarchy import dendrogram

INSPECT_K = 30    # the k you want to take forward and group
N_GROUPS  = 6      # how many merged pathway groups you want to end up with

with open(C.MODELS_DIR / f"kmeans_k{INSPECT_K}.pkl", "rb") as f:
    km_i = pickle.load(f)
cent_deg = P.centroids_to_degrees(km_i.cluster_centers_)   # for readable labels
group_map, Z = P.suggest_groups(km_i.cluster_centers_, N_GROUPS)

fig, ax = plt.subplots(figsize=(11, 4))
dendrogram(Z, labels=[f"{c}" for c in range(INSPECT_K)], ax=ax,
           color_threshold=Z[-(N_GROUPS-1), 2])
ax.set_title(f"Centroid dendrogram (k={INSPECT_K}); cut -> {N_GROUPS} groups")
ax.set_xlabel("raw cluster id"); ax.set_ylabel("Ward distance")
# fig.savefig(C.FIG_DIR / f"centroid_dendrogram_k{INSPECT_K}.png", dpi=130, bbox_inches="tight")
plt.show()

print("Suggested GROUP_MAP (paste into config.py, then refine):")
print("GROUP_MAP =", group_map)
for g in sorted(set(group_map.values())):
    members = [c for c, gg in group_map.items() if gg == g]
    print(f"  group {g}: clusters {members}")

Suggested GROUP_MAP (paste into config.py, then refine):
GROUP_MAP = {0: 1, 1: 5, 2: 1, 3: 3, 4: 4, 5: 0, 6: 3, 7: 5, 8: 5, 9: 1, 10: 1, 11: 0, 12: 2, 13: 3, 14: 1, 15: 4, 16: 2, 17: 5, 18: 3, 19: 1, 20: 5, 21: 3, 22: 5, 23: 3, 24: 3, 25: 4, 26: 0, 27: 5, 28: 5, 29: 1}
  group 0: clusters [5, 11, 26]
  group 1: clusters [0, 2, 9, 10, 14, 19, 29]
  group 2: clusters [12, 16]
  group 3: clusters [3, 6, 13, 18, 21, 23, 24]
  group 4: clusters [4, 15, 25]
  group 5: clusters [1, 7, 8, 17, 20, 22, 27, 28]


## 2.5b  Numbered centroid map (eyeball the grouping)

Companion to the dendrogram: each raw cluster's **number** sits on its day-100 centroid, the line shows its day50→day100 drift, and the number's **halo colour is the group `suggest_groups` proposes**. Clusters whose numbers sit close with parallel lines are the ones to merge — refine `GROUP_MAP` from what you see here.

In [7]:
# --- Numbered centroid map: eyeball the grouping alongside the dendrogram ---
import matplotlib.pyplot as plt, matplotlib.patheffects as pe
from matplotlib.lines import Line2D

cl_cmap   = plt.get_cmap("tab20", INSPECT_K)
grp_ids   = sorted(set(group_map.values()))
grp_cmap  = plt.get_cmap("Set1", max(len(grp_ids), 3))
grp_color = {g: grp_cmap(i) for i, g in enumerate(grp_ids)}

fig, ax = plt.subplots(figsize=(8, 7))
for c in range(INSPECT_K):
    lat50, lon50, lat100, lon100 = cent_deg[c]
    ax.plot([lon50, lon100], [lat50, lat100], "-", color=cl_cmap(c), lw=1.5, alpha=.7)
    ax.scatter(lon50,  lat50,  s=15, color=cl_cmap(c))                        # day 50 (start)
    ax.scatter(lon100, lat100, s=45, color=cl_cmap(c), edgecolor="k", lw=.5, zorder=3)  # day 100 (end)
    # the cluster number, haloed in its SUGGESTED group colour -> see what merges
    ax.text(lon100, lat100, str(c), fontsize=11, fontweight="bold",
            ha="center", va="center", color="white", zorder=4,
            path_effects=[pe.withStroke(linewidth=3, foreground=grp_color[group_map[c]])])

ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.set_title(f"Centroids day50→day100 (k={INSPECT_K}); halo = suggested group "
             f"({N_GROUPS}). Compare with the dendrogram.")
ax.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=grp_color[g], mec="k",
                          label=f"group {g}") for g in grp_ids], fontsize=8)
# fig.savefig(C.FIG_DIR / f"centroid_map_k{INSPECT_K}.png", dpi=130, bbox_inches="tight")
plt.show()

In [8]:
# --- Numbered centroid MAP: eyeball the grouping alongside the dendrogram ---
import matplotlib.pyplot as plt, matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import cartopy.crs as ccrs, cartopy.feature as cfeature

cl_cmap   = plt.get_cmap("tab20", INSPECT_K)
grp_ids   = sorted(set(group_map.values()))
grp_cmap  = plt.get_cmap("Set1", max(len(grp_ids), 3))
grp_color = {g: grp_cmap(i) for i, g in enumerate(grp_ids)}

pc = ccrs.PlateCarree()
fig = plt.figure(figsize=(8, 7))
ax = plt.axes(projection=pc)
ax.set_extent([-80, -30,
              -5, 20], crs=pc)
ax.add_feature(cfeature.LAND, facecolor="0.85"); ax.coastlines(lw=.5)
gl = ax.gridlines(draw_labels=True, lw=.3, color="0.7", alpha=.5)
gl.top_labels = gl.right_labels = False

for c in range(INSPECT_K):
    lat50, lon50, lat100, lon100 = cent_deg[c]
    ax.plot([lon50, lon100], [lat50, lat100], "-", color=cl_cmap(c), lw=1.5,
            alpha=.7, transform=pc)
    ax.scatter(lon50,  lat50,  s=15, color=cl_cmap(c), transform=pc)                       # day 50 (start)
    ax.scatter(lon100, lat100, s=45, color=cl_cmap(c), edgecolor="k", lw=.5,
               zorder=3, transform=pc)                                                     # day 100 (end)
    # the cluster number, haloed in its SUGGESTED group colour -> see what merges
    ax.text(lon100, lat100, str(c), fontsize=11, fontweight="bold",
            ha="center", va="center", color="white", zorder=4, transform=pc,
            path_effects=[pe.withStroke(linewidth=3, foreground=grp_color[group_map[c]])])

ax.set_title(f"Centroids day50→day100 (k={INSPECT_K}); halo = suggested group "
             f"({N_GROUPS}). Compare with the dendrogram.")
ax.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=grp_color[g], mec="k",
                          label=f"group {g}") for g in grp_ids], fontsize=8, loc="best")
fig.savefig(C.FIG_DIR / f"centroid_map_k{INSPECT_K}.png", dpi=130, bbox_inches="tight")
plt.show()


## 2.6  Summary

Models for every k are pickled in `data/kmeans_models/`. Decide:
1. which **k** to carry forward (set `BEST_K` in notebooks 03 & 04), and
2. (optionally) the **GROUP_MAP** that merges fine clusters into pathway groups
   (paste into `config.py`).

If you set `GROUP_MAP`, notebooks 03 and 04 will additionally produce a
`cluster_group` column and group-level figures; if you leave it empty they fall
back to the raw clusters.

In [9]:
print("models saved:", sorted(p.name for p in C.MODELS_DIR.glob('kmeans_k*.pkl')))
print(res.to_string(index=False))

models saved: ['kmeans_k10.pkl', 'kmeans_k15.pkl', 'kmeans_k20.pkl', 'kmeans_k25.pkl', 'kmeans_k30.pkl', 'kmeans_k35.pkl']
 k    inertia  silhouette  min_size  max_size
15 300.116215    0.392295      2254     59886
20 256.164544    0.358026       964     61368
25 217.522165    0.400126      1519     46002
30 193.798016    0.396741      1230     46101
35 183.053062    0.355265      1185     31346
